In [0]:
%run ./utils

# Load

In [0]:
dbutils.widgets.text("silver_table_name", "tb_canais")
silver_table_name = dbutils.widgets.get("silver_table_name")

df_canais = (
    spark.table(f"{silver_schema}.{silver_table_name}")
        .distinct()
        .orderBy("id_canal")
)

display(df_canais)

# Tratamento

In [0]:
df_dim_canal = add_sequential_key(df_canais, "canal_sk")

In [0]:
df_dim_canal = (
    df_dim_canal
        .select(
            "canal_sk",
            "id_canal",
            "nome_canal",
            "tipo_canal"
        )
)

display(df_dim_canal)

# Verificações

In [0]:
# A dimensão deve ter uma linha por canal e a chave deve ser única
total = df_dim_canal.count()
canals = df_dim_canal.select("id_canal").distinct().count()
chaves = df_dim_canal.select("canal_sk").distinct().count()
print(f"Linhas: {total} | id_canal únicos: {canals} | sk_canal únicas: {chaves}")

# Salvamento

In [0]:
dbutils.widgets.text("gold_table_name", "dim_canal")
gold_table_name = dbutils.widgets.get("gold_table_name")

save_df(df_dim_canal, gold_table_name)